# Retrieval smoke test

Top-5 results from each retrieval leg (dense / BM25) for 10 hand-picked
queries — three monolingual per language plus deliberately cross-lingual ones
(Ukrainian question about an English regulation, etc.). Sanity check, not an
evaluation: the eval harness with metrics lands in Phase 8.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # make `src` importable from notebooks/

In [ ]:
from src.ingestion.chunker import load_chunks
from src.retrieval.bm25_index import BM25Index
from src.retrieval.embedding import get_embedder
from src.retrieval import qdrant_store

chunks = {c.chunk_id: c for c in load_chunks("child")}
bm25 = BM25Index.load()
embedder = get_embedder()
client = qdrant_store.get_client()
print("chunks:", len(chunks))

In [ ]:
QUERIES = [
    ("en", "What are the lawful bases for processing personal data?"),
    ("en", "Which AI practices are prohibited under the AI Act?"),
    ("en", "What obligations do very large online platforms have?"),
    ("pl", "Jakie kary pieniężne przewiduje RODO za naruszenia?"),
    ("pl", "Ile dni urlopu wypoczynkowego przysługuje pracownikowi?"),
    ("pl", "Czy można trenować modele AI na danych osobowych?"),
    ("uk", "Які права має суб'єкт персональних даних?"),
    # deliberately cross-lingual: answers live in another language's documents
    ("uk", "Які системи штучного інтелекту заборонені в ЄС?"),
    ("pl", "Jakie wymogi muszą spełniać systemy AI wysokiego ryzyka?"),
    ("en", "What does Ukrainian law require for consent to data processing?"),
]

def show(results, label):
    print(f"  {label}")
    for cid, score in results[:5]:
        c = chunks[cid]
        print(f"    {score:6.3f}  [{c.language}] {c.source_id:24} {c.ref or c.kind}")

for lang, q in QUERIES:
    print(f"\n=== [{lang}] {q}")
    dense = qdrant_store.search(client, embedder.embed_query(q), top_k=5)
    show(dense, "dense (e5 + qdrant)")
    show(bm25.search(q, top_k=5), "bm25")